In [1]:
from dotenv import load_dotenv
import pandas as pd
from langchain_openai import OpenAIEmbeddings

load_dotenv("../../.env")

True

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/2040_seoul_plan.pdf")
pages = loader.load_and_split()
print('청크의 수:', len(pages))

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8056\3216081774.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


청크의 수: 193


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(pages)

print("분할된 청크의 수:", len(splits))

분할된 청크의 수: 293


In [5]:
from langchain_chroma import Chroma
import os

persist_directory = '../chroma_store'

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )

Creating new Chroma store


In [6]:
print('문서의 수:', vectorstore._collection.count())

문서의 수: 293


In [7]:
# similarity_search 메서드 사용
question = "미래교통수단의 상용화"
top_three_docs = vectorstore.similarity_search(question, k=3)

for i, doc in enumerate(top_three_docs, 1):
    print(f"문서 {i}:")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")
    print('--' * 20)

문서 1:
내용: 주차장, 정류소 등 기존 교통시설과 접목
Ÿ 기존의 교통수단과 자율주행자동차, UAM, 마이크로모빌리티 등 새롭게 등장하는 다양한 교통수단을 
통합하여 관제하고 운영·관리할 수 있는 기반을 조성한다.
Ÿ 대중교통 중심의 통합 모빌리티 플랫폼을 구축하여 다양한 이동 수단...
메타데이터: {'title': '', 'file_path': '../data/2040_seoul_plan.pdf', 'creationdate': '2024-12-12T18:16:11+09:00', 'trapped': '', 'subject': '', 'moddate': '2024-12-12T18:16:11+09:00', 'format': 'PDF 1.4', 'producer': 'Hancom PDF 1.3.0.542', 'modDate': "D:20241212181611+09'00'", 'page': 97, 'source': '../data/2040_seoul_plan.pdf', 'total_pages': 205, 'author': 'SI', 'creationDate': "D:20241212181611+09'00'", 'creator': 'Hwp 2020 11.0.0.5178', 'keywords': ''}
----------------------------------------
문서 2:
내용: 54
제2장 미래상과 목표
5. 기술발전에 선제적 대응, ‘미래교통 인프라 구축’
1) 배경
미래교통수단 도입에 따른 도시 활동 변화 및 공간구조 재편 대비
Ÿ 빠르게 진화하는 미래 신 교통수단은 도시공간에 새로운 변화를 일으킬 것이며, 도시 내 이동패턴의 
변화를 가...
메타데이터: {'producer': 'Hancom PDF 1.3.0.542', 'file_path': '../data/2040_seoul_plan.pdf', 'creationDate': "D:20241212181611+09'00'", 'source': '../data/2040_seoul_plan.pdf', '